In [12]:
from pathlib import Path
import json
import re
import unicodedata

In [13]:
EXCERPT_ID = "excerpt_000"

PROJECT_ROOT = Path("/workspace/storyfx")

SEGMENT_SPEC_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "composition_plans"
    / f"pred_{EXCERPT_ID}_segment_specification.json"
)

FA_PARSED_PATH = (
    PROJECT_ROOT
    / "data"
    / EXCERPT_ID
    / "alignment"
    / f"{EXCERPT_ID}_fa_parsed.txt"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "schedules"
    / f"{EXCERPT_ID}_segment_schedule.json"
)


print("Segment specification:", SEGMENT_SPEC_PATH)
print("Forced alignment:", FA_PARSED_PATH)
print("Output:", OUTPUT_PATH)

Segment specification: \workspace\storyfx\outputs\composition_plans\pred_excerpt_000_segment_specification.json
Forced alignment: \workspace\storyfx\data\excerpt_000\alignment\excerpt_000_fa_parsed.txt
Output: \workspace\storyfx\outputs\schedules\excerpt_000_segment_schedule.json


In [14]:
def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)

    text = (
        text
        .replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    return text.lower()

In [15]:
def lexical_tokens(text):
    """
    Convert text into lexical tokens.

    Examples:
        "rosy-coloured" -> ["rosy", "coloured"]
        "don't" -> ["don't"]
        "room," -> ["room"]
    """
    text = normalize_text(text)

    return re.findall(
        r"[a-z0-9]+(?:'[a-z0-9]+)?",
        text,
    )

In [16]:
def flatten_fa_words(fa_words):
    flat_tokens = []
    word_indices = []

    for i, word in enumerate(fa_words):
        tokens = lexical_tokens(word["word"])

        for token in tokens:
            flat_tokens.append(token)
            word_indices.append(i)

    return flat_tokens, word_indices

In [17]:
def load_fa_words(path):
    text = Path(path).read_text(
        encoding="utf-8"
    ).strip()

    words = []

    for item in text.split():
        if "|" not in item:
            continue

        word, end_time = item.rsplit("|", 1)

        words.append(
            {
                "word": word,
                "normalized": normalize_word(word),
                "end_time": float(end_time),
            }
        )

    # Infer each word's start time from the preceding word's end.
    for i, word in enumerate(words):
        if i == 0:
            word["start_time"] = 0.0
        else:
            word["start_time"] = words[i - 1]["end_time"]

    return words

In [18]:
fa_words = load_fa_words(
    FA_PARSED_PATH
)

print("Aligned words:", len(fa_words))
print(fa_words[:10])

FileNotFoundError: [Errno 2] No such file or directory: '\\workspace\\storyfx\\data\\excerpt_000\\alignment\\excerpt_000_fa_parsed.txt'

In [7]:
def anchor_tokens(anchor):
    tokens = []

    for word in anchor.split():
        normalized = normalize_word(word)

        if normalized:
            tokens.append(normalized)

    return tokens

In [8]:
def find_anchor(
    anchor,
    fa_words,
    start_index=0,
):
    target = lexical_tokens(anchor)

    if not target:
        raise ValueError(
            f"Anchor contains no matchable words: {anchor}"
        )

    flat_tokens, word_indices = flatten_fa_words(
        fa_words
    )

    # Convert the MFA word-level start index
    # into a flattened-token start position.
    flat_start = 0

    while (
        flat_start < len(word_indices)
        and word_indices[flat_start] < start_index
    ):
        flat_start += 1

    target_length = len(target)

    for i in range(
        flat_start,
        len(flat_tokens) - target_length + 1,
    ):
        candidate = flat_tokens[
            i:i + target_length
        ]

        if candidate == target:
            first_word_index = word_indices[i]

            last_word_index = word_indices[
                i + target_length - 1
            ]

            return (
                first_word_index,
                last_word_index,
            )

    raise ValueError(
        "Could not match anchor:\n"
        f"{anchor}\n\n"
        f"Normalized tokens:\n{target}"
    )

In [9]:
def quantify_segments(
    segment_spec_path,
    fa_parsed_path,
):
    segments = json.loads(
        Path(segment_spec_path).read_text(
            encoding="utf-8"
        )
    )

    fa_words = load_fa_words(
        fa_parsed_path
    )

    scheduled_segments = []

    search_index = 0

    for segment in segments:
        start_anchor = segment["start"]
        end_anchor = segment["end"]

        # --------------------------------------------
        # Match start anchor
        # --------------------------------------------

        start_first, start_last = find_anchor(
            start_anchor,
            fa_words,
            start_index=search_index,
        )

        # --------------------------------------------
        # Match end anchor
        # --------------------------------------------

        end_first, end_last = find_anchor(
            end_anchor,
            fa_words,
            start_index=start_first,
        )

        if end_last < start_first:
            raise ValueError(
                f"End anchor occurs before start anchor "
                f"for {segment['segment_id']}"
            )

        start_time = fa_words[
            start_first
        ]["start_time"]

        end_time = fa_words[
            end_last
        ]["end_time"]

        quantified = {
            **segment,
            "start_time": round(
                start_time,
                2,
            ),
            "end_time": round(
                end_time,
                2,
            ),
        }

        scheduled_segments.append(
            quantified
        )

        # Future segments must occur after this one.
        search_index = end_last + 1

    return scheduled_segments

In [ ]:
def diagnose_anchor(anchor, fa_words):
    target = lexical_tokens(anchor)

    print("Anchor tokens:")
    print(target)

    if not target:
        return

    first = target[0]

    print(
        f"\nOccurrences of first token '{first}':"
    )

    for i, word in enumerate(fa_words):
        if first in lexical_tokens(word["word"]):
            context = fa_words[
                max(0, i - 5):i + 15
            ]

            print(
                i,
                [w["word"] for w in context]
            )

In [ ]:
segment_schedule = quantify_segments(
    SEGMENT_SPEC_PATH,
    FA_PARSED_PATH,
)

segment_schedule

In [ ]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)
1
OUTPUT_PATH.write_text(
    json.dumps(
        segment_schedule,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    f"Segment schedule saved to: {OUTPUT_PATH}"
)